# PKLot parking occupancy - train on Colab

**Before you start**
1. Upload this whole project folder to Google Drive as `MyDrive/Pklot`. You can skip `.venv`, `archive` and `data`.
2. `Runtime -> Change runtime type -> GPU` (A100/L4 is best for YOLOv8x; a T4 works but is slower).

Trained weights are copied to `MyDrive/Pklot/weights/` at the end of each run. The original notebooks
saved the wrong folder to Drive and the weights were lost, so check that folder exists before closing the tab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Pklot'   # source code lives here, weights are saved here
# Work from local disk: reading thousands of images through Drive during training is very slow.
!rm -rf /content/Pklot && mkdir -p /content/Pklot
!rsync -a --exclude data --exclude runs --exclude output --exclude .venv --exclude archive "{PROJECT}/" /content/Pklot/
%cd /content/Pklot
!pip install -q -r requirements.txt

## YOLOv8 (the model the report selected)
Report recipe: `yolov8x`, 50 epochs, batch 20, 352 px. About 1 hour on an A100 (it took 1.07 h originally).

In [ ]:
!python -m pklot.train_yolo --model yolov8x.pt --epochs 50 --batch 20 --imgsz 352 --cache ram --save-dir "{PROJECT}/weights"

## Detectron2 baselines: Faster R-CNN and RetinaNet
Detectron2 has no pip release; building from source is the fragile step. If it fails on the current Colab image, the
build recipe that worked in 2024 is in `archive/notebooks/Detectron_T4_bs5_10k.ipynb` (first cells).

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
!python -m pklot.train_detectron --model faster_rcnn --save-dir "{PROJECT}/weights"

In [ ]:
!python -m pklot.train_detectron --model retinanet --save-dir "{PROJECT}/weights"

## Check that the weights were saved

In [ ]:
!ls -R "{PROJECT}/weights" | head -50